# Integration and Plotting

This tutorial covers integrating orbits in potentials, plotting their
trajectories, accessing orbital quantities, and working with
non-inertial reference frames.

In [ ]:
%matplotlib inline
import numpy
from astropy import units
from galpy.orbit import Orbit
from galpy.potential import MWPotential2014, LogarithmicHaloPotential

## Basic integration

Use `o.integrate(ts, pot)` to integrate an orbit. The time array `ts`
should start at 0 (the initial condition).

In [ ]:
o = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])
ts = numpy.linspace(0.0, 10.0, 10000)
o.integrate(ts, MWPotential2014)
print("Integration complete. Final R =", o.R(ts[-1]))

## Automatic time determination

You can pass just the potential and galpy will integrate for ~10 dynamical times.

In [ ]:
o_auto = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])
o_auto.integrate(MWPotential2014)
o_auto.plot()

## Physical units for time

When using `ro=` and `vo=`, you can pass time arrays in physical units.

In [ ]:
o_phys = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0], ro=8.0, vo=220.0)
ts_phys = numpy.linspace(0.0, 5.0, 5000) * units.Gyr
o_phys.integrate(ts_phys, MWPotential2014)
print("R at 5 Gyr =", o_phys.R(ts_phys[-1]))

## Parallel integration of multiple orbits

Multi-orbit objects are integrated in parallel using C.

In [ ]:
numpy.random.seed(42)
vxvvs = numpy.column_stack(
    [
        numpy.random.uniform(0.8, 1.2, 100),
        numpy.random.normal(0.0, 0.05, 100),
        numpy.random.uniform(0.9, 1.1, 100),
        numpy.random.normal(0.0, 0.05, 100),
        numpy.random.normal(0.0, 0.05, 100),
        numpy.random.uniform(0.0, 2 * numpy.pi, 100),
    ]
)
os = Orbit(vxvvs)
ts = numpy.linspace(0.0, 10.0, 1001)
os.integrate(ts, MWPotential2014)
print("All", os.size, "orbits integrated.")
print("R shape at all times:", os.R(ts).shape)

## Continuing integrations

You can continue an integration forward or backward by providing a time
array that starts where the previous one ended.

In [ ]:
o_cont = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])

# Forward integration
ts_fwd = numpy.linspace(0.0, 5.0, 5000)
o_cont.integrate(ts_fwd, MWPotential2014)

# Continue forward
ts_ext = numpy.linspace(5.0, 10.0, 5000)
o_cont.integrate(ts_ext, MWPotential2014)

# Backward integration from t=0
ts_bwd = numpy.linspace(0.0, -5.0, 5000)
o_cont.integrate(ts_bwd, MWPotential2014)

print("Time range now covers t = -5 to 10")
o_cont.plot()

## Displaying orbits: various projections

Use `o.plot()` with `d1` and `d2` to select projections.

In [ ]:
o = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])
ts = numpy.linspace(0.0, 10.0, 10000)
o.integrate(ts, MWPotential2014)

In [ ]:
# Default: R vs. z (meridional plane)
o.plot()

In [ ]:
# Face-on view: x vs. y
o.plot(d1="x", d2="y")

In [ ]:
# Phase space: R vs. vR
o.plot(d1="R", d2="vR")

In [ ]:
# Sky coordinates (requires ro/vo)
o_sky = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0], ro=8.0, vo=220.0)
o_sky.integrate(ts, MWPotential2014)
o_sky.plot(d1="ll", d2="bb")

In [ ]:
o_sky.plot(d1="ra", d2="dec")

## Custom expression plotting

You can use arbitrary expressions with numexpr syntax.

In [ ]:
o.plot(d1="t", d2="R*cos(phi)")

## Orbit characterization

After integration, compute orbital parameters numerically.

In [ ]:
print("Eccentricity:", o.e())
print("Apocenter:", o.rap())
print("Pericenter:", o.rperi())
print("Max |z|:", o.zmax())

In [ ]:
# Analytic computation using the Staeckel approximation (no integration needed)
o_new = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])
print("Analytic e:", o_new.e(analytic=True, pot=MWPotential2014, type="staeckel"))
print("Analytic rap:", o_new.rap(analytic=True, pot=MWPotential2014, type="staeckel"))

## Energy and energy conservation

In [ ]:
print("Energy at t=0:", o.E(0.0))
print("Energy at t=10:", o.E(ts[-1]))
print("Relative energy error:", (o.E(ts[-1]) - o.E(0.0)) / abs(o.E(0.0)))

## Accessing raw orbital data

Evaluate orbital quantities at any time, or get the full array.

In [ ]:
# At specific time
print("R(t=5):", o.R(5.0))
print("phi(t=5):", o.phi(5.0))

# Full orbit array: shape (ntimes, ndim)
orbit_array = o.getOrbit()
print("Full orbit shape:", orbit_array.shape)

In [ ]:
# Sky coordinates at specific time (requires ro/vo)
print("RA(t=5):", o_sky.ra(5.0))
print("Dec(t=5):", o_sky.dec(5.0))

## Creating a new orbit from evaluated position

Calling an orbit as a function returns a new Orbit at that time.

In [ ]:
o_at_5 = o(5.0)
print("New orbit at t=5:", o_at_5)
print("R:", o_at_5.R())

## Non-inertial frames

galpy supports orbit integration in non-inertial reference frames
using `NonInertialFrameForce`. Here is a brief example of a
rotating frame.

In [ ]:
from galpy.potential import NonInertialFrameForce

# Set up a frame rotating at the LSR angular frequency
lp = LogarithmicHaloPotential(normalize=1.0)
nif = NonInertialFrameForce(Omega=1.0)  # Omega = vcirc/R = 1 in natural units

o_rot = Orbit([1.0, 0.1, 1.0, 0.0, 0.1, 0.0])
ts = numpy.linspace(0.0, 20.0, 20000)
o_rot.integrate(ts, lp + nif)
o_rot.plot(d1="x", d2="y")